# JOINs

In [2]:
import psycopg
import pandas as pd
conn = psycopg.connect(
    host = 'localhost',
    port= 5432,
    dbname ='chinook',
    user = 'postgres',
    password = '231165'
)

cur = conn.cursor()

## INNER JOIN

In [3]:
# Match customers with their invoices


result = pd.read_sql('''
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    i.invoice_id,
    i.total
FROM customer AS c
INNER JOIN invoice AS i
    ON c.customer_id = i.customer_id
ORDER BY i.total DESC
LIMIT 10;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\3567955572.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,first_name,last_name,invoice_id,total
0,6,Helena,Holý,404,25.86
1,26,Richard,Cunningham,299,23.86
2,45,Ladislav,Kovács,96,21.86
3,46,Hugh,O'Reilly,194,21.86
4,25,Victor,Stevens,201,18.86
5,7,Astrid,Gruber,89,18.86
6,57,Luis,Rojas,88,17.91
7,5,František,Wichterlová,306,16.86
8,43,Isabelle,Mercier,313,16.86
9,4,Bjørn,Hansen,208,15.86


In [4]:
# Display album titles with artist names


result = pd.read_sql('''
SELECT
    al.title AS album_title,
    ar.name AS artist_name
FROM album AS al
INNER JOIN artist AS ar
    ON al.artist_id = ar.artist_id
ORDER BY ar.name, al.title
LIMIT 15;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\80594211.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,album_title,artist_name
0,"A Copland Celebration, Vol. I",Aaron Copland & London Symphony Orchestra
1,Worlds,Aaron Goldberg
2,For Those About To Rock We Salute You,AC/DC
3,Let There Be Rock,AC/DC
4,The World of Classical Favourites,Academy of St. Martin in the Fields & Sir Nevi...
5,Sir Neville Marriner: A Celebration,Academy of St. Martin in the Fields Chamber En...
6,"Fauré: Requiem, Ravel: Pavane & Others","Academy of St. Martin in the Fields, John Birc..."
7,Bach: Orchestral Suites Nos. 1 - 4,"Academy of St. Martin in the Fields, Sir Nevil..."
8,Balls to the Wall,Accept
9,Restless and Wild,Accept


## LEFT JOIN

In [5]:
# Include all customers, even without invoices


result = pd.read_sql('''
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    i.invoice_id,
    i.total
FROM customer AS c
LEFT JOIN invoice AS i
    ON c.customer_id = i.customer_id
ORDER BY c.customer_id;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\1841947134.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,first_name,last_name,invoice_id,total
0,1,Luís,Gonçalves,382,8.91
1,1,Luís,Gonçalves,327,13.86
2,1,Luís,Gonçalves,98,3.98
3,1,Luís,Gonçalves,316,1.98
4,1,Luís,Gonçalves,195,0.99
...,...,...,...,...,...
407,59,Puja,Srivastava,23,3.96
408,59,Puja,Srivastava,229,13.86
409,59,Puja,Srivastava,45,5.94
410,59,Puja,Srivastava,97,1.99


In [6]:
# Find customers without invoices


result = pd.read_sql('''
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
LEFT JOIN invoice AS i
    ON c.customer_id = i.customer_id
WHERE i.invoice_id IS NULL;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\3256289777.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,first_name,last_name


## Joining Multiplt Tables

In [8]:
# Combine tracks, albums, and artists

result = pd.read_sql('''
SELECT
    t.name AS track_name,
    al.title AS album_title,
    ar.name AS artist_name
FROM track AS t
INNER JOIN album AS al
    ON t.album_id = al.album_id
INNER JOIN artist AS ar
    ON al.artist_id = ar.artist_id
ORDER BY ar.name, al.title, t.name
LIMIT 20;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\3087725991.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,track_name,album_title,artist_name
0,Fanfare for the Common Man,"A Copland Celebration, Vol. I",Aaron Copland & London Symphony Orchestra
1,OAM's Blues,Worlds,Aaron Goldberg
2,Breaking The Rules,For Those About To Rock We Salute You,AC/DC
3,C.O.D.,For Those About To Rock We Salute You,AC/DC
4,Evil Walks,For Those About To Rock We Salute You,AC/DC
5,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC
6,Inject The Venom,For Those About To Rock We Salute You,AC/DC
7,Let's Get It Up,For Those About To Rock We Salute You,AC/DC
8,Night Of The Long Knives,For Those About To Rock We Salute You,AC/DC
9,Put The Finger On You,For Those About To Rock We Salute You,AC/DC


## Agrregatin with JOIN

In [10]:
# Calculate total spending for each customer

result = pd.read_sql('''
SELECT
    c.first_name,
    c.last_name,
    SUM(i.total) AS total_spent
FROM customer AS c
INNER JOIN invoice AS i
    ON c.customer_id = i.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
ORDER BY total_spent DESC;
''', conn)

result

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\1580243075.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,first_name,last_name,total_spent
0,Helena,Holý,49.62
1,Richard,Cunningham,47.62
2,Luis,Rojas,46.62
3,Ladislav,Kovács,45.62
4,Hugh,O'Reilly,45.62
5,Julia,Barnett,43.62
6,Fynn,Zimmermann,43.62
7,Frank,Ralston,43.62
8,Astrid,Gruber,42.62
9,Victor,Stevens,42.62


In [12]:
# Count tracks for each artist

result = pd.read_sql('''
SELECT
    ar.name AS artist_name,
    COUNT(t.track_id) AS track_count
FROM artist AS ar
INNER JOIN album AS al
    ON ar.artist_id = al.artist_id
INNER JOIN track AS t
    ON al.album_id = t.album_id
GROUP BY
    ar.artist_id,
    ar.name
ORDER BY track_count DESC
LIMIT 10;
''', conn)

result

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\675260668.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,artist_name,track_count
0,Iron Maiden,213
1,U2,135
2,Led Zeppelin,114
3,Metallica,112
4,Deep Purple,92
5,Lost,92
6,Pearl Jam,67
7,Lenny Kravitz,57
8,Various Artists,56
9,The Office,53


## FULL OUTER JOIN

In [13]:
# Keep matched and unmatched rows from both tables

result = pd.read_sql('''
SELECT
    al.title AS album_title,
    ar.name AS artist_name
FROM album AS al
FULL OUTER JOIN artist AS ar
    ON al.artist_id = ar.artist_id;
''', conn)

result

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_10004\186569842.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,album_title,artist_name
0,For Those About To Rock We Salute You,AC/DC
1,Balls to the Wall,Accept
2,Restless and Wild,Accept
3,Let There Be Rock,AC/DC
4,Big Ones,Aerosmith
...,...,...
413,NaN,Pedro Luís & A Parede
414,NaN,Matisyahu
415,NaN,Mundo Livre S/A
416,NaN,Santana Feat. Lauryn Hill & Cee-Lo
